
# Exporting FLIM images to TIFF (no ImageJ)
``tttrlib`` can write the images it computes from a TTTR/FLIM measurement
straight to standard TIFF files, so you can archive them or open them in any
image viewer **without ImageJ or any other extra software**. Reading TIFFs back
into NumPy is just as direct.

The whole TIFF layer is backed by a *bundled* libtiff (statically linked), so it
adds no runtime dependency to ``tttrlib``.

Two calls do everything:

* ``tttrlib.imwrite(path, array)`` - write a 2-D image or a 3-D ``(frames,
  height, width)`` stack. The array's dtype selects the on-disk pixel type
  (``uint8/16/32``, ``int32``, ``float32/64``); ``compression`` may be
  ``"none"``, ``"lzw"`` (default), ``"packbits"`` or ``"deflate"``.
* ``tttrlib.imread(path)`` - read a TIFF back, auto-detecting the pixel type and
  returning a NumPy array (2-D for a single page, 3-D for a stack).

A TIFF is a flat sequence of pages, which is a problem as soon as a measurement
has both frames *and* colours: six pages cannot say whether they are six time
points or two time points in three channels. ``imwrite(..., axes="TCYX")``
records that split as ImageJ hyperstack metadata, ``imread`` restores the shape,
and ``tiff_metadata(path)`` reports it without decoding any pixels.


In [ ]:
import os
import sys
import tempfile
from pathlib import Path

import numpy as np
import pylab as plt

import tttrlib
# Make the `examples` package importable when this script is run directly,
# from any working directory.
sys.path[:0] = [str(_p) for _p in Path(__file__).resolve().parents
                if (_p / "examples" / "_example_data.py").is_file()][:1]
from examples._example_data import get_data_path

## Build FLIM images from a TTTR measurement
Read a confocal laser-scanning (CLSM) measurement and map the photon stream to
pixels. From the filled image we derive two per-pixel images:

* the **intensity** image - a ``(frames, lines, pixels)`` ``uint16`` photon
  count stack, and
* a **mean micro time** image - a floating-point map that (without an IRF)
  already reflects the fluorescence lifetime contrast.



In [ ]:
data = tttrlib.TTTR(str(get_data_path('imaging/pq/ht3/pq_ht3_clsm.ht3')), 'HT3')

clsm = tttrlib.CLSMImage(data, fill=True, channels=(0, 1))

# 3-D intensity stack (one 2-D image per scanned frame), native uint16
intensity_stack = clsm.intensity
# 2-D intensity image summed over all frames (uint32 so counts never overflow)
intensity_sum = intensity_stack.sum(axis=0).astype(np.uint32)

# 2-D mean-micro-time image (float32), frames stacked into one
mean_micro_time = clsm.get_mean_micro_time(
    tttr_data=data,
    minimum_number_of_photons=3,
    stack_frames=True
).astype(np.float32)[0]

print("intensity stack :", intensity_stack.shape, intensity_stack.dtype)
print("intensity sum   :", intensity_sum.shape, intensity_sum.dtype)
print("mean micro time :", mean_micro_time.shape, mean_micro_time.dtype)

## Write the images to TIFF
A 3-D array becomes a multi-page TIFF (one page per frame); a 2-D array becomes
a single-page TIFF. The dtype is preserved on disk, so the float lifetime map
stays floating point and the integer intensities stay integer.



In [ ]:
out_dir = Path(tempfile.mkdtemp(prefix="tttrlib_flim_tiff_"))

tttrlib.imwrite(out_dir / "intensity_stack.tif", intensity_stack, compression="lzw")
tttrlib.imwrite(out_dir / "intensity_sum.tif", intensity_sum, compression="lzw")
tttrlib.imwrite(out_dir / "mean_micro_time.tif", mean_micro_time, compression="lzw")

for f in sorted(out_dir.glob("*.tif")):
    print(f"{f.name:24s} {os.path.getsize(f):>8d} bytes")

## Read the TIFFs back
``imread`` auto-detects the pixel type. A single-page file returns a 2-D array;
the multi-page stack returns a 3-D array. The round-trip is exact.



In [ ]:
stack_back = tttrlib.imread(out_dir / "intensity_stack.tif")
sum_back = tttrlib.imread(out_dir / "intensity_sum.tif")
tau_back = tttrlib.imread(out_dir / "mean_micro_time.tif")

print("stack round-trip exact :", np.array_equal(stack_back, intensity_stack))
print("sum   round-trip exact :", np.array_equal(sum_back, intensity_sum))
print("tau   round-trip exact :", np.array_equal(tau_back, mean_micro_time))
print("read-back dtypes       :", stack_back.dtype, sum_back.dtype, tau_back.dtype)

# tiff_info / tiff_dtype expose the on-disk geometry and pixel type
info = tttrlib.tiff_info(str(out_dir / "intensity_stack.tif"))
print("stack on disk          :",
      (info.n_frames, info.height, info.width),
      tttrlib.tiff_dtype(str(out_dir / "intensity_stack.tif")))

## Keeping frames and colours apart: a hyperstack
The measurement has two detection channels. Filling one CLSM image per channel
gives a ``(frames, channels, lines, pixels)`` array - four dimensions, where a
plain TIFF only has pages. Naming the axes on write stores the split, so the
file reads back as the same 4-D array instead of as ``frames x channels``
anonymous pages, and ImageJ opens it as a hyperstack with a channel slider.



In [ ]:
per_channel = np.stack(
    [tttrlib.CLSMImage(data, fill=True, channels=(ch,)).intensity for ch in (0, 1)],
    axis=1,  # (frames, channels, lines, pixels)
)
print("per-channel stack:", per_channel.shape, per_channel.dtype)

hyperstack_path = out_dir / "intensity_hyperstack.tif"
tttrlib.imwrite(hyperstack_path, per_channel, axes="TCYX")

meta = tttrlib.tiff_metadata(hyperstack_path)
print("axes on disk     :", meta["axes"], meta["shape"], meta["dtype"])

hyperstack_back = tttrlib.imread(hyperstack_path)
print("shape preserved  :", hyperstack_back.shape == per_channel.shape)
print("round-trip exact :", np.array_equal(hyperstack_back, per_channel))

# Without the axis labels the same pixels would come back as one flat page axis:
print("pages on disk    :", tttrlib.tiff_info(str(hyperstack_path)).n_frames)

## Visualise what was exported
These are exactly the arrays that were written to (and read back from) the TIFF
files - the figure below is rendered from the data ``imread`` returned.



In [ ]:
mask = sum_back < 3  # hide near-empty pixels in the lifetime map
masked_tau = np.ma.masked_where(mask, tau_back)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].set_title("Intensity (sum of frames)\nintensity_sum.tif")
im0 = ax[0].imshow(sum_back, cmap="cividis")
fig.colorbar(im0, ax=ax[0], fraction=0.046)

ax[1].set_title(f"Intensity stack, frame 0\nintensity_stack.tif ({stack_back.shape[0]} pages)")
im1 = ax[1].imshow(stack_back[0], cmap="cividis")
fig.colorbar(im1, ax=ax[1], fraction=0.046)

ax[2].set_title("Mean micro time\nmean_micro_time.tif")
im2 = ax[2].imshow(masked_tau, cmap="Spectral")
fig.colorbar(im2, ax=ax[2], fraction=0.046)

for a in ax:
    a.set_xticks([])
    a.set_yticks([])
plt.tight_layout()
plt.show()

That's it: the FLIM intensity and lifetime images are now standard TIFF files
on disk (``%s``) that open in any viewer - no ImageJ required.



In [ ]:
print("TIFF files written to:", out_dir)